# Phúc — Random Forest, cảnh báo và đánh giá

**Chạy từng cell từ trên xuống. Lần đầu chỉ làm phần A.**

Nguồn chính: BDG2. Bộ này giúp hoàn thành phần AI riêng trước khi nối vào Kafka của nhóm. Mô hình ước lượng mức tiêu thụ kỳ vọng tại giờ đang quan sát; nó chưa phải dự báo tương lai với thời tiết chưa biết.

- A: kiểm tra code bằng dữ liệu **giả**; không dùng các số này trong báo cáo.
- B: huấn luyện và đánh giá bằng **dữ liệu thật** từ Gold hoặc ASHRAE.
- C: giao model cho Dương và chuẩn bị số đo benchmark.

Môi trường được dùng để kiểm tra bộ code: Python 3.12, Java 17, PySpark 3.5.6. Nếu kernel khác, cell cài đặt tạo môi trường Python 3.12 riêng khi cần. Nếu project nhóm dùng Spark khác, thử trong môi trường riêng trước và thống nhất phiên bản với Dương.

In [ ]:
#@title 1. Kiểm tra Python và Java
import sys, os, subprocess, shutil, json, time
from pathlib import Path
from IPython.display import display, Image, Markdown
print("Python kernel:", sys.version.split()[0])
if shutil.which("java"):
    result = subprocess.run(["java", "-version"], capture_output=True, text=True)
    print(result.stderr or result.stdout)
else:
    print("Chưa có Java; cell tiếp theo sẽ cài Java 17 trong Colab/Linux nếu có quyền.")
ROOT = (Path("/content/phuc_bigdata") if Path("/content").is_dir()
        else Path.cwd() / "phuc_bigdata")
ROOT.mkdir(parents=True, exist_ok=True)
print("Thư mục chạy:", ROOT)

In [ ]:
#@title 2. Nạp code và cài môi trường (chạy một lần) {display-mode: "form"}
import base64, io, zipfile, re
PAYLOAD = '...'
with zipfile.ZipFile(io.BytesIO(base64.b64decode(PAYLOAD))) as z:
    z.extractall(ROOT)
RUN_PYTHON = sys.executable
if not (3, 10) <= sys.version_info[:2] <= (3, 12):
    env_python = ROOT / ".phuc_env" / ("Scripts/python.exe" if os.name == "nt" else "bin/python")
    if not env_python.is_file():
        print("Tạo môi trường Python 3.12 riêng cho Spark; không đổi kernel.", flush=True)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
        subprocess.run([sys.executable, "-m", "uv", "python", "install", "3.12"], check=True)
        subprocess.run([sys.executable, "-m", "uv", "venv", "--seed", "--python", "3.12",
                        str(ROOT / ".phuc_env")], check=True)
    RUN_PYTHON = str(env_python)
java_info = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr if shutil.which("java") else ""
if not re.search(r'version "17[.\"]', java_info):
    if os.name == "posix" and shutil.which("apt-get") and hasattr(os, "geteuid") and os.geteuid() == 0:
        subprocess.run(["apt-get", "update", "-qq"], check=True)
        subprocess.run(["apt-get", "install", "-y", "-qq", "openjdk-17-jre-headless"], check=True)
        candidates = sorted(Path("/usr/lib/jvm").glob("java-17-openjdk-*"))
        if candidates:
            os.environ["JAVA_HOME"] = str(candidates[0])
            os.environ["PATH"] = str(candidates[0] / "bin") + os.pathsep + os.environ["PATH"]
    else:
        raise RuntimeError("Cài Java 17 và đặt JAVA_HOME, rồi chạy lại cell này; hoặc dùng Docker trong START_HERE.md.")
if shutil.which("java"):
    java_bin = Path(shutil.which("java")).resolve()
    os.environ["JAVA_HOME"] = str(java_bin.parent.parent)
ENV = os.environ.copy()
for key in ["PYTHONHOME", "PYTHONPATH", "SPARK_HOME"]:
    ENV.pop(key, None)
ENV["SPARK_LOCAL_IP"] = "127.0.0.1"
ENV["MPLBACKEND"] = "Agg"
ENV["TZ"] = "UTC"
ENV["PYSPARK_PYTHON"] = RUN_PYTHON
ENV["PYSPARK_DRIVER_PYTHON"] = RUN_PYTHON

def run(*args):
    print("Đang chạy:", " ".join(map(str, args)), flush=True)
    process = subprocess.Popen([RUN_PYTHON, "-u", *map(str, args)], cwd=ROOT, env=ENV,
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, encoding="utf-8", errors="replace", bufsize=1)
    lines = []
    for line in process.stdout:
        print(line, end="", flush=True)
        lines.append(line)
    status = process.wait()
    if status:
        error_file = ROOT / "last_error.log"
        error_file.write_text("".join(lines), encoding="utf-8")
        raise RuntimeError(f"Lệnh dừng với mã {status}. Lỗi gốc đã hiện phía trên; cũng được lưu ở {error_file}.")

run("-m", "pip", "install", "-q", "-r", ROOT / "requirements.txt")
# Import pyspark alone is insufficient: importing ML also checks distutils.
run("-c", "import sys, pyspark, pandas, matplotlib; import distutils.version; from pyspark.ml import Pipeline; "
    "print('Python chạy Spark:', sys.version.split()[0]); "
    "print('PySpark:', pyspark.__version__); print('Matplotlib backend:', matplotlib.get_backend()); print('Import Spark ML: OK')")
active_java = subprocess.run(["java", "-version"], env=ENV, capture_output=True, text=True)
print(active_java.stderr or active_java.stdout)
if active_java.returncode:
    active_java.check_returncode()
print("Sẵn sàng. Python chạy Spark:", RUN_PYTHON)


## A. Chạy thử code

Dữ liệu ở phần này được sinh bằng công thức, không phải ASHRAE. Kết quả cao hoặc thấp đều không chứng minh chất lượng trên dữ liệu thật. Mỗi lần chạy dùng thư mục mới để tránh ghi đè.

In [ ]:
#@title 3. Sinh dữ liệu giả và huấn luyện thử
DEMO = ROOT / "results" / ("demo_" + time.strftime("%Y%m%d_%H%M%S"))
run("phuc_analytics.py", "demo-data", "--output", "data/demo.csv", "--days", "14", "--buildings", "3")
run("phuc_analytics.py", "train", "--input", "data/demo.csv", "--output", DEMO,
    "--origin", "synthetic", "--trees", "12", "--depth", "5")
LAST_RESULT = DEMO
print("Xong bước thử:", DEMO)

In [ ]:
#@title 4. Xem kết quả thử — chỉ để hiểu các file
metrics = json.loads((DEMO / "metrics.json").read_text())
print("NGUỒN DỮ LIỆU:", metrics["data_origin"])
print(json.dumps(metrics, ensure_ascii=False, indent=2))
for name in ["actual_predicted.png", "feature_importance.png", "residuals.png"]:
    display(Image(filename=str(DEMO / name)))
display(Markdown("**Nếu có bảng và biểu đồ thì phần huấn luyện đã chạy. Tiếp theo lấy dữ liệu thật; chưa chép các số này vào báo cáo.**"))

In [ ]:
#@title 5. Thử detector với các phép đo sửa có chủ đích
ANOMALY_DEMO = ROOT / "results" / ("anomaly_demo_" + time.strftime("%Y%m%d_%H%M%S"))
run("phuc_analytics.py", "injection-test", "--input", DEMO / "heldout_test",
    "--model", DEMO / "model", "--output", ANOMALY_DEMO)
display(Image(filename=str(ANOMALY_DEMO / "confusion_matrix.png")))

## B. Dùng dữ liệu thật BDG2

Nguồn của đề tài: [Building Data Genome 2](https://github.com/buds-lab/building-data-genome-project-2).

**Đã chạy demo rồi thì mở bản này, chạy cell 1–2 và chuyển thẳng tới cell 6–8.** Môi trường và thư mục kết quả đã có được tái sử dụng.

Cell 6 mặc định tải điện RAW, metadata và weather từ BDG2, ghép thành Gold. Khoảng 200 MB dữ liệu nguồn; cần giữ kết nối trong lúc tải. Dùng RAW để giữ số 0 và tăng vọt; điện RAW đã có đơn vị kWh.

Phạm vi pilot mặc định: site Robin nếu có, tối đa 10 tòa nhà, 180 ngày từ 01/01/2016. File provenance ghi đúng phạm vi thực chạy. Mã tòa nhà/site dạng chữ và giờ địa phương được giữ nhất quán.

Nếu đã có BDG2 trên Colab, chọn `bdg2_local` và điền DATA_PATH. Nếu dùng Gold của Dung, chọn `gold_csv`/`gold_parquet`, điền đường dẫn và đơn vị. `raw_ashrae` chỉ dùng cho bản Kaggle có train.csv, building_metadata.csv và weather_train.csv.


In [ ]:
#@title 6. Chuẩn bị dữ liệu thật BDG2
MODE = "bdg2_download" #@param ["bdg2_download", "bdg2_local", "gold_csv", "gold_parquet", "raw_ashrae"]
DATA_PATH = "" #@param {type:"string"}
GOLD_UNIT = "CHUA_XAC_NHAN" #@param {type:"string"}
BDG2_SITE = "" #@param {type:"string"}
START_DATE = "2016-01-01" #@param {type:"string"}
SITE_ID = 1 #@param {type:"integer"}
BUILDING_LIMIT = 10 #@param {type:"integer"}
MAX_DAYS = 180 #@param {type:"integer"}
if MODE != "bdg2_download" and (not DATA_PATH or not Path(DATA_PATH).exists()):
    raise ValueError("Upload/gắn dữ liệu trước rồi điền DATA_PATH là đường dẫn tồn tại.")
if MODE.startswith("gold_") and GOLD_UNIT == "CHUA_XAC_NHAN":
    raise ValueError("Xác nhận đơn vị Gold với Dung rồi điền GOLD_UNIT, ví dụ kWh.")
STAMP = time.strftime("%Y%m%d_%H%M%S")
REAL = ROOT / "results" / ("real_" + STAMP)
if MODE in ["bdg2_download", "bdg2_local"]:
    raw_dir = ROOT / "data" / "bdg2" if MODE == "bdg2_download" else Path(DATA_PATH)
    GOLD = ROOT / "data" / ("bdg2_gold_" + STAMP + ".csv")
    arguments = ["prepare_bdg2.py", "--raw-dir", raw_dir, "--output", GOLD,
                 "--building-limit", BUILDING_LIMIT, "--max-days", MAX_DAYS,
                 "--start", START_DATE]
    if MODE == "bdg2_download":
        arguments.append("--download")
    if BDG2_SITE.strip():
        arguments += ["--site-id", BDG2_SITE.strip()]
    run(*arguments)
    INPUT, UNIT = GOLD, "kWh"
    PROVENANCE = GOLD.with_suffix(".provenance.json")
elif MODE == "raw_ashrae":
    GOLD = ROOT / "data" / ("gold_" + STAMP)
    run("phuc_analytics.py", "prepare", "--raw-dir", DATA_PATH, "--output", GOLD,
        "--site-id", SITE_ID, "--building-limit", BUILDING_LIMIT, "--max-days", MAX_DAYS)
    INPUT, UNIT = GOLD, "kWh"
    PROVENANCE = Path(str(GOLD) + "_provenance.json")
else:
    INPUT, UNIT = Path(DATA_PATH), GOLD_UNIT
    PROVENANCE = None
print("Đầu vào:", INPUT, "Đơn vị:", UNIT)
print("Kết quả sẽ ở:", REAL)


In [ ]:
#@title 7. Huấn luyện và đánh giá dữ liệu thật
run("phuc_analytics.py", "train", "--input", INPUT, "--output", REAL,
    "--origin", "real", "--unit", UNIT, "--trees", "60", "--depth", "8")
if PROVENANCE is not None:
    shutil.copyfile(PROVENANCE, REAL / "data_provenance.json")
LAST_RESULT = REAL
real_metrics = json.loads((REAL / "metrics.json").read_text())
print(json.dumps(real_metrics, ensure_ascii=False, indent=2))
for name in ["actual_predicted.png", "feature_importance.png", "residuals.png"]:
    display(Image(filename=str(REAL / name)))

In [ ]:
#@title 8. Đánh giá kịch bản bất thường trên tập test thật
ANOMALY_REAL = ROOT / "results" / ("anomaly_real_" + time.strftime("%Y%m%d_%H%M%S"))
run("phuc_analytics.py", "injection-test", "--input", REAL / "heldout_test",
    "--model", REAL / "model", "--output", ANOMALY_REAL)
print(json.dumps(json.loads((ANOMALY_REAL / "injection_metrics.json").read_text()), ensure_ascii=False, indent=2))
display(Image(filename=str(ANOMALY_REAL / "confusion_matrix.png")))
display(Markdown("**Nhãn ở đây là phép đo đã sửa / chưa sửa. Dữ liệu gốc không có nhãn xác nhận sự cố; không gọi các chỉ số này là độ chính xác phát hiện sự cố thực tế.**"))

## C. Giao kết quả và nối vào project của nhóm

- Gửi Dương `model/`, `phuc_analytics.py` và `streaming_hooks.py` từ bộ ZIP. Dương dùng `AnalyticsBundle.score()` bên trong `foreachBatch` sau khi gộp metadata và thời tiết.
- Mục 4.5 dùng `metrics.json`, biểu đồ Actual–Predicted, residual và feature importance của **lần chạy real**.
- Mục 4.6 dùng ngưỡng và kết quả kịch bản có kiểm soát, ghi đúng giới hạn nhãn.
- Mục 4.7.3 cần ảnh dashboard của nhóm sau tích hợp; notebook riêng chưa chứng minh Kafka/MongoDB/dashboard chạy.
- Mục 4.8 cần số đo nhiều mức tải. Hùng cung cấp rate, batch size theo byte, partition, số message ACK và thời gian; Dương cung cấp progress và backlog. Thống nhất `run_id`, `event_id`, `produced_at_ms` và điểm cuối đo trước khi chạy.

Xem `START_HERE.md` và `NOI_DUNG_BAO_CAO.md` trong bộ ZIP để viết đúng từng mục. Bộ này chưa truy cập được repo trong PDF nên chưa gắn trực tiếp vào source của nhóm.

In [ ]:
#@title 9. Tải kết quả của lần chạy gần nhất
archive = shutil.make_archive(str(ROOT / ("ket_qua_" + LAST_RESULT.name)), "zip", root_dir=LAST_RESULT)
print("Đã tạo:", archive)
if LAST_RESULT.name.startswith("demo"):
    print("Đây là KẾT QUẢ DEMO; không dùng để nộp số liệu thực nghiệm.")
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print("Mở thư mục notebook để lấy file ZIP ở đường dẫn trên.")
if "ANOMALY_REAL" in globals() and ANOMALY_REAL.exists():
    anomaly_archive = shutil.make_archive(str(ROOT / ("ket_qua_" + ANOMALY_REAL.name)), "zip", root_dir=ANOMALY_REAL)
    print("Kết quả kịch bản có kiểm soát:", anomaly_archive)
    try:
        files.download(anomaly_archive)
    except NameError:
        pass